In [3]:
# ============================================================
# NB04 — Baseline Texto (MLP sobre embeddings MiniLM)
# ============================================================
# Objetivo: treinar um MLP simples sobre os 384-dim embeddings
# de texto gerados no NB02, para ter um baseline unimodal
# textual comparável ao baseline tabular do NB03 (ROC-AUC 0.676).
#
# Nota: usamos sklearn MLPClassifier em vez de Keras/TF para
# evitar conflitos de dependências NumPy 2.x vs TF 2.16.
# Para um baseline simples sobre embeddings já calculados,
# a diferença de performance é negligível.

import numpy as np
import pandas as pd
from pathlib import Path
import json

# Modelo
from sklearn.neural_network import MLPClassifier

# Métricas (mesmas do NB03 para comparação direta)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report
)

# Reprodutibilidade
SEED = 42
np.random.seed(SEED)

# Configuração de paths
PROJECT_ROOT = Path.cwd().parent  # notebooks/ → raiz do projeto
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_CACHE = PROJECT_ROOT / "data" / "cache"
RESULTS_DIR = PROJECT_ROOT / "results" / "nb04_baseline_texto"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"\nResultados serão guardados em: {RESULTS_DIR}")

NumPy version: 2.4.4
Pandas version: 3.0.2

Resultados serão guardados em: /Users/rennandamiani/Documents/ISAG/ProjetoFinal_PetFinder/results/nb04_baseline_texto


In [4]:
# ============================================================
# Célula 2 — Carregar splits + embeddings de texto e alinhar
# ============================================================

# 1. Carregar o mapa central de splits (gerado no NB02)
splits = pd.read_parquet(DATA_PROCESSED / "splits_index.parquet")
print(f"splits_index shape: {splits.shape}")
print(f"Colunas: {splits.columns.tolist()}")
print(f"\nDistribuição por split:")
print(splits['split'].value_counts())
print(f"\nPrimeiras linhas:")
splits.head()

splits_index shape: (14993, 4)
Colunas: ['PetID', 'split', 'y', 'RescuerID']

Distribuição por split:
split
train    9600
test     2997
val      2396
Name: count, dtype: int64

Primeiras linhas:


,PetID,split,y,RescuerID
0,6296e909a,train,1,3082c7125d8fb66f7dd4bff4192c8b14
1,5842f1ff5,train,1,9238e4f44c71a75282e62f7136c6b240
2,850a43f90,train,1,95481e953f8aed9ec3d16fc4509537e8
3,d24c30b4b,train,1,22fe332bf9c924d4718005891c63fbed
4,1caa6fcdb,train,1,1e0b5a458b5b77f5af581d57ebf570b3


In [5]:
# ============================================================
# Célula 3 — Carregar embeddings MiniLM + PetIDs
# ============================================================

# Embeddings: matriz (14993, 384) de floats
text_embeddings = np.load(DATA_CACHE / "text_embeddings_minilm.npy")

# PetIDs: array (14993,) de strings — diz-nos a que animal corresponde cada linha
text_petids = np.load(DATA_CACHE / "text_embeddings_petids.npy", allow_pickle=True)

print(f"text_embeddings shape: {text_embeddings.shape}")
print(f"text_embeddings dtype: {text_embeddings.dtype}")
print(f"text_petids shape: {text_petids.shape}")
print(f"\nPrimeiros 3 PetIDs: {text_petids[:3]}")
print(f"Primeiros 5 valores do primeiro embedding:\n{text_embeddings[0, :5]}")


text_embeddings shape: (14993, 384)
text_embeddings dtype: float32
text_petids shape: (14993,)

Primeiros 3 PetIDs: ['6296e909a' '5842f1ff5' '850a43f90']
Primeiros 5 valores do primeiro embedding:
[0.01334256 0.01349591 0.05370471 0.04412433 0.06771659]


In [6]:
# ============================================================
# Célula 4 — Alinhar embeddings ao splits_index pelo PetID
# ============================================================

# 1. Criar um dicionário {PetID -> índice da linha no array de embeddings}
#    Exemplo: {'abc123': 0, 'def456': 1, ...}
petid_to_idx = {pid: i for i, pid in enumerate(text_petids)}

# 2. Para cada PetID em splits_index (pela ordem do splits!), buscar o índice
#    correspondente no array de embeddings
embedding_indices = splits['PetID'].map(petid_to_idx).values

# 3. Verificar se há algum PetID em splits que não existe nos embeddings
n_missing = pd.isna(embedding_indices).sum()
print(f"PetIDs em splits sem embedding correspondente: {n_missing}")
assert n_missing == 0, "Há PetIDs sem embedding! Investigar antes de continuar."

# 4. Reordenar a matriz de embeddings para bater certo com splits
text_embeddings_aligned = text_embeddings[embedding_indices.astype(int)]

print(f"\nShape original embeddings: {text_embeddings.shape}")
print(f"Shape após alinhamento:    {text_embeddings_aligned.shape}")
print(f"Shape do splits_index:     {splits.shape}")

# 5. Verificação de sanidade: PetID na linha 0 do splits tem de bater com
#    o PetID que originalmente estava no índice 0 reordenado
sample_petid = splits.iloc[0]['PetID']
sample_embedding_idx = petid_to_idx[sample_petid]
print(f"\nVerificação:")
print(f"  Primeiro PetID em splits: {sample_petid}")
print(f"  Índice original nos embeddings: {sample_embedding_idx}")
print(f"  Embedding alinhado[0] == embedding original[{sample_embedding_idx}]?",
      np.array_equal(text_embeddings_aligned[0], text_embeddings[sample_embedding_idx]))

PetIDs em splits sem embedding correspondente: 0

Shape original embeddings: (14993, 384)
Shape após alinhamento:    (14993, 384)
Shape do splits_index:     (14993, 4)

Verificação:
  Primeiro PetID em splits: 6296e909a
  Índice original nos embeddings: 0
  Embedding alinhado[0] == embedding original[0]? True


In [7]:
# ============================================================
# Célula 5 — Criar X_train, X_val, X_test e respetivos y
# ============================================================

# Máscaras booleanas para cada split
train_mask = (splits['split'] == 'train').values
val_mask   = (splits['split'] == 'val').values
test_mask  = (splits['split'] == 'test').values

# Features (X): embeddings de texto, 384 dimensões
X_train = text_embeddings_aligned[train_mask]
X_val   = text_embeddings_aligned[val_mask]
X_test  = text_embeddings_aligned[test_mask]

# Labels (y): target binário AdoptionFast
y_train = splits.loc[train_mask, 'y'].values
y_val   = splits.loc[val_mask,   'y'].values
y_test  = splits.loc[test_mask,  'y'].values

# PetIDs (para guardar com as previsões no final)
petids_train = splits.loc[train_mask, 'PetID'].values
petids_val   = splits.loc[val_mask,   'PetID'].values
petids_test  = splits.loc[test_mask,  'PetID'].values

print("Shapes:")
print(f"  X_train: {X_train.shape}   y_train: {y_train.shape}")
print(f"  X_val:   {X_val.shape}   y_val:   {y_val.shape}")
print(f"  X_test:  {X_test.shape}   y_test:  {y_test.shape}")

print("\nBalanceamento (% classe 1 = Adoção Rápida):")
print(f"  Train: {y_train.mean():.4f}")
print(f"  Val:   {y_val.mean():.4f}")
print(f"  Test:  {y_test.mean():.4f}")

Shapes:
  X_train: (9600, 384)   y_train: (9600,)
  X_val:   (2396, 384)   y_val:   (2396,)
  X_test:  (2997, 384)   y_test:  (2997,)

Balanceamento (% classe 1 = Adoção Rápida):
  Train: 0.5027
  Val:   0.5029
  Test:  0.5025


In [8]:
# ============================================================
# Célula 6 — Definir e treinar o MLP sobre embeddings de texto
# ============================================================

# Arquitetura: 384 (input) → 128 → 64 → 1 (sigmoid implícito via classificação)
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),   # duas camadas escondidas
    activation='relu',               # função de ativação não-linear
    solver='adam',                   # otimizador (variante de SGD, bom default)
    alpha=1e-4,                      # regularização L2 (evita overfitting)
    batch_size=64,                   # tamanho do minibatch
    learning_rate_init=1e-3,         # taxa de aprendizagem inicial
    max_iter=100,                    # número máximo de épocas
    early_stopping=True,             # parar cedo se o val interno parar de melhorar
    validation_fraction=0.1,         # 10% do train vai para val interno (monitorização)
    n_iter_no_change=10,             # paciência: 10 épocas sem melhorar → para
    random_state=SEED,
    verbose=True                     # imprimir progresso
)

print("Arquitetura do MLP:")
print(f"  Input:           {X_train.shape[1]} dims (embedding MiniLM)")
print(f"  Hidden layer 1:  128 neurónios (ReLU)")
print(f"  Hidden layer 2:  64 neurónios (ReLU)")
print(f"  Output:          1 (logistic)")
print(f"\nA treinar com {X_train.shape[0]} amostras...\n")

# Treinar
mlp.fit(X_train, y_train)

print(f"\nTreino concluído em {mlp.n_iter_} épocas.")
print(f"Best validation score (interno): {mlp.best_validation_score_:.4f}")

Arquitetura do MLP:
  Input:           384 dims (embedding MiniLM)
  Hidden layer 1:  128 neurónios (ReLU)
  Hidden layer 2:  64 neurónios (ReLU)
  Output:          1 (logistic)

A treinar com 9600 amostras...

Iteration 1, loss = 0.68884852
Validation score: 0.586458
Iteration 2, loss = 0.67514105
Validation score: 0.588542
Iteration 3, loss = 0.66345392
Validation score: 0.611458
Iteration 4, loss = 0.64601119
Validation score: 0.597917
Iteration 5, loss = 0.62670241
Validation score: 0.601042
Iteration 6, loss = 0.59751330
Validation score: 0.603125
Iteration 7, loss = 0.56149837
Validation score: 0.602083
Iteration 8, loss = 0.52262723
Validation score: 0.612500
Iteration 9, loss = 0.48260338
Validation score: 0.604167
Iteration 10, loss = 0.43379710
Validation score: 0.576042
Iteration 11, loss = 0.38936801
Validation score: 0.588542
Iteration 12, loss = 0.35422607
Validation score: 0.578125
Iteration 13, loss = 0.31728108
Validation score: 0.584375
Iteration 14, loss = 0.27424968

In [9]:
# ============================================================
# Célula 7 — Avaliar no validation set (comparação com NB03)
# ============================================================

# 1. Previsões probabilísticas (para ROC-AUC, PR-AUC)
y_val_proba = mlp.predict_proba(X_val)[:, 1]  # coluna 1 = prob de classe "Rápida"

# 2. Previsões de classe (threshold default = 0.5)
y_val_pred = (y_val_proba >= 0.5).astype(int)

# 3. Calcular todas as métricas (mesmas do NB03)
metrics_val = {
    'accuracy':  accuracy_score(y_val, y_val_pred),
    'precision': precision_score(y_val, y_val_pred),
    'recall':    recall_score(y_val, y_val_pred),
    'f1':        f1_score(y_val, y_val_pred),
    'roc_auc':   roc_auc_score(y_val, y_val_proba),
    'pr_auc':    average_precision_score(y_val, y_val_proba),
}

print("=" * 55)
print("NB04 — Baseline Texto (MLP) — Validation set")
print("=" * 55)
for name, value in metrics_val.items():
    print(f"  {name:10s}: {value:.4f}")

# 4. Matriz de confusão
print("\nMatriz de confusão (threshold=0.5):")
cm = confusion_matrix(y_val, y_val_pred)
print(f"  TN={cm[0,0]:4d}   FP={cm[0,1]:4d}")
print(f"  FN={cm[1,0]:4d}   TP={cm[1,1]:4d}")

# 5. Comparação direta com NB03
print("\n" + "=" * 55)
print("Comparação com NB03 (baseline tabular LightGBM)")
print("=" * 55)
nb03_roc = 0.6758
diff = metrics_val['roc_auc'] - nb03_roc
print(f"  NB03 (tabular):  ROC-AUC = {nb03_roc:.4f}")
print(f"  NB04 (texto):    ROC-AUC = {metrics_val['roc_auc']:.4f}")
print(f"  Diferença:       {diff:+.4f} ({'texto melhor' if diff > 0 else 'tabular melhor'})")

NB04 — Baseline Texto (MLP) — Validation set
  accuracy  : 0.5188
  precision : 0.5221
  recall    : 0.5095
  f1        : 0.5157
  roc_auc   : 0.5312
  pr_auc    : 0.5249

Matriz de confusão (threshold=0.5):
  TN= 629   FP= 562
  FN= 591   TP= 614

Comparação com NB03 (baseline tabular LightGBM)
  NB03 (tabular):  ROC-AUC = 0.6758
  NB04 (texto):    ROC-AUC = 0.5312
  Diferença:       -0.1446 (tabular melhor)


In [10]:
# ============================================================
# Célula 7b — Diagnóstico: está tudo alinhado? o sinal é real?
# ============================================================

# ---- Teste 1: distribuição das probabilidades previstas ----
# Se o modelo estiver "confuso", as probas vão amontoar-se perto de 0.5
import numpy as np
print("Distribuição das probabilidades previstas no val:")
print(f"  min:    {y_val_proba.min():.4f}")
print(f"  25%:    {np.percentile(y_val_proba, 25):.4f}")
print(f"  50%:    {np.percentile(y_val_proba, 50):.4f}")
print(f"  75%:    {np.percentile(y_val_proba, 75):.4f}")
print(f"  max:    {y_val_proba.max():.4f}")
print(f"  std:    {y_val_proba.std():.4f}")

# ---- Teste 2: comparar com previsão no próprio train ----
# Se o modelo acerta bem no train mas falha no val → overfitting
y_train_proba = mlp.predict_proba(X_train)[:, 1]
train_roc = roc_auc_score(y_train, y_train_proba)
print(f"\nROC-AUC no train:   {train_roc:.4f}")
print(f"ROC-AUC no val:     {metrics_val['roc_auc']:.4f}")
print(f"Gap train-val:      {train_roc - metrics_val['roc_auc']:+.4f}")

# ---- Teste 3: sanity check de alinhamento ----
# Pegar 3 animais aleatórios no val e confirmar que tudo bate certo
print("\nSanity check de alinhamento (3 animais aleatórios do val):")
val_indices_in_splits = np.where(val_mask)[0]
rng = np.random.default_rng(SEED)
sample_positions = rng.choice(len(val_indices_in_splits), size=3, replace=False)

for pos in sample_positions:
    # Posição no val
    petid_from_splits = petids_val[pos]
    y_from_splits = y_val[pos]
    # Vai ao splits original confirmar
    row = splits[splits['PetID'] == petid_from_splits].iloc[0]
    # Vai aos embeddings originais confirmar
    orig_emb_idx = petid_to_idx[petid_from_splits]
    embeddings_match = np.array_equal(X_val[pos], text_embeddings[orig_emb_idx])
    print(f"  PetID {petid_from_splits}: "
          f"y_val={y_from_splits}, splits.y={row['y']}, "
          f"split={row['split']}, embeddings_match={embeddings_match}")

Distribuição das probabilidades previstas no val:
  min:    0.0000
  25%:    0.0789
  50%:    0.4747
  75%:    0.8695
  max:    1.0000
  std:    0.3742

ROC-AUC no train:   0.9588
ROC-AUC no val:     0.5312
Gap train-val:      +0.4275

Sanity check de alinhamento (3 animais aleatórios do val):
  PetID ce4484c3d: y_val=0, splits.y=0, split=val, embeddings_match=True
  PetID 385028a00: y_val=1, splits.y=1, split=val, embeddings_match=True
  PetID 22e1f3438: y_val=1, splits.y=1, split=val, embeddings_match=True


In [11]:
# ============================================================
# Célula 8 — MLP v2: rede menor + regularização forte
# ============================================================
# O v1 overfitou brutalmente (train 0.96, val 0.53).
# Reduzimos capacidade e reforçamos regularização.

mlp_v2 = MLPClassifier(
    hidden_layer_sizes=(64, 32),    # ↓ de (128, 64): menos capacidade
    activation='relu',
    solver='adam',
    alpha=0.01,                      # ↑ de 1e-4: 100x mais regularização L2
    batch_size=64,
    learning_rate_init=1e-3,
    max_iter=100,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=SEED,
    verbose=False                    # menos ruído no output desta vez
)

print("MLP v2 — arquitetura e regularização reforçada:")
print(f"  Input:           {X_train.shape[1]}")
print(f"  Hidden layer 1:  64 (↓ de 128)")
print(f"  Hidden layer 2:  32 (↓ de 64)")
print(f"  alpha (L2):      0.01 (↑ de 1e-4)")
print(f"\nA treinar...\n")

mlp_v2.fit(X_train, y_train)
print(f"Treino concluído em {mlp_v2.n_iter_} épocas.\n")

# Avaliar em train e val simultaneamente
y_train_proba_v2 = mlp_v2.predict_proba(X_train)[:, 1]
y_val_proba_v2   = mlp_v2.predict_proba(X_val)[:, 1]
y_val_pred_v2    = (y_val_proba_v2 >= 0.5).astype(int)

train_roc_v2 = roc_auc_score(y_train, y_train_proba_v2)
val_roc_v2   = roc_auc_score(y_val, y_val_proba_v2)

print("=" * 55)
print("Comparação v1 vs v2")
print("=" * 55)
print(f"                  v1        v2")
print(f"  ROC-AUC train:  0.9588    {train_roc_v2:.4f}")
print(f"  ROC-AUC val:    0.5312    {val_roc_v2:.4f}")
print(f"  Gap:            +0.4275   {train_roc_v2 - val_roc_v2:+.4f}")

# Métricas completas para o v2 (para decidir se é o nosso modelo final)
metrics_val_v2 = {
    'accuracy':  accuracy_score(y_val, y_val_pred_v2),
    'precision': precision_score(y_val, y_val_pred_v2),
    'recall':    recall_score(y_val, y_val_pred_v2),
    'f1':        f1_score(y_val, y_val_pred_v2),
    'roc_auc':   val_roc_v2,
    'pr_auc':    average_precision_score(y_val, y_val_proba_v2),
}
print(f"\nMétricas v2 no val:")
for name, value in metrics_val_v2.items():
    print(f"  {name:10s}: {value:.4f}")

MLP v2 — arquitetura e regularização reforçada:
  Input:           384
  Hidden layer 1:  64 (↓ de 128)
  Hidden layer 2:  32 (↓ de 64)
  alpha (L2):      0.01 (↑ de 1e-4)

A treinar...

Treino concluído em 16 épocas.

Comparação v1 vs v2
                  v1        v2
  ROC-AUC train:  0.9588    0.6849
  ROC-AUC val:    0.5312    0.5464
  Gap:            +0.4275   +0.1385

Métricas v2 no val:
  accuracy  : 0.5275
  precision : 0.5332
  recall    : 0.4871
  f1        : 0.5091
  roc_auc   : 0.5464
  pr_auc    : 0.5427


In [12]:
# ============================================================
# Célula 9 — MLP v3: arquitetura mínima + regularização agressiva
# ============================================================
# Hipótese: o modelo precisa de pouca capacidade. Uma única
# camada escondida pequena + alpha muito alto força a rede a
# aprender apenas os padrões mais robustos.

mlp_v3 = MLPClassifier(
    hidden_layer_sizes=(32,),       # apenas UMA camada escondida, com 32 neurónios
    activation='relu',
    solver='adam',
    alpha=0.05,                      # 5x mais forte que o v2, 500x mais que o v1
    batch_size=64,
    learning_rate_init=1e-3,
    max_iter=100,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=SEED,
    verbose=False
)

print("MLP v3 — mínimo viável:")
print(f"  Input:           {X_train.shape[1]}")
print(f"  Hidden layer 1:  32 (única camada)")
print(f"  alpha (L2):      0.05 (↑ de 0.01)")
print(f"\nA treinar...\n")

mlp_v3.fit(X_train, y_train)
print(f"Treino concluído em {mlp_v3.n_iter_} épocas.\n")

# Avaliar
y_train_proba_v3 = mlp_v3.predict_proba(X_train)[:, 1]
y_val_proba_v3   = mlp_v3.predict_proba(X_val)[:, 1]
y_val_pred_v3    = (y_val_proba_v3 >= 0.5).astype(int)

train_roc_v3 = roc_auc_score(y_train, y_train_proba_v3)
val_roc_v3   = roc_auc_score(y_val, y_val_proba_v3)

# Métricas completas
metrics_val_v3 = {
    'accuracy':  accuracy_score(y_val, y_val_pred_v3),
    'precision': precision_score(y_val, y_val_pred_v3),
    'recall':    recall_score(y_val, y_val_pred_v3),
    'f1':        f1_score(y_val, y_val_pred_v3),
    'roc_auc':   val_roc_v3,
    'pr_auc':    average_precision_score(y_val, y_val_proba_v3),
}

print("=" * 65)
print("Comparação final: v1 vs v2 vs v3")
print("=" * 65)
print(f"                         v1        v2        v3")
print(f"  Arquitetura:       (128,64)  (64,32)    (32,)")
print(f"  alpha:              0.0001    0.01      0.05")
print(f"  ROC-AUC train:      0.9588    0.6849    {train_roc_v3:.4f}")
print(f"  ROC-AUC val:        0.5312    0.5464    {val_roc_v3:.4f}")
print(f"  Gap train-val:     +0.4275   +0.1385   {train_roc_v3 - val_roc_v3:+.4f}")

print(f"\nMétricas v3 no val:")
for name, value in metrics_val_v3.items():
    print(f"  {name:10s}: {value:.4f}")

# Matriz de confusão v3
cm_v3 = confusion_matrix(y_val, y_val_pred_v3)
print(f"\nMatriz de confusão v3:")
print(f"  TN={cm_v3[0,0]:4d}   FP={cm_v3[0,1]:4d}")
print(f"  FN={cm_v3[1,0]:4d}   TP={cm_v3[1,1]:4d}")

MLP v3 — mínimo viável:
  Input:           384
  Hidden layer 1:  32 (única camada)
  alpha (L2):      0.05 (↑ de 0.01)

A treinar...

Treino concluído em 12 épocas.

Comparação final: v1 vs v2 vs v3
                         v1        v2        v3
  Arquitetura:       (128,64)  (64,32)    (32,)
  alpha:              0.0001    0.01      0.05
  ROC-AUC train:      0.9588    0.6849    0.6008
  ROC-AUC val:        0.5312    0.5464    0.5469
  Gap train-val:     +0.4275   +0.1385   +0.0539

Métricas v3 no val:
  accuracy  : 0.5196
  precision : 0.5297
  recall    : 0.3992
  f1        : 0.4553
  roc_auc   : 0.5469
  pr_auc    : 0.5361

Matriz de confusão v3:
  TN= 764   FP= 427
  FN= 724   TP= 481


In [13]:
# ============================================================
# Célula 10 — v3 no test set + tabela comparativa dos 3 modelos
# ============================================================

# 1. Previsões do v3 no test set
y_test_proba = mlp_v3.predict_proba(X_test)[:, 1]
# Nota: NÃO calculamos y_test_pred aqui.
# A decisão do threshold fica para o NB06 (fusão multimodal),
# seguindo a mesma convenção do NB03.

# 2. DataFrames de previsões (mesmo formato do NB03)
val_predictions = pd.DataFrame({
    'PetID':  petids_val,
    'y_true': y_val,
    'y_proba': y_val_proba_v3,
    'y_pred': y_val_pred_v3,
})

test_predictions = pd.DataFrame({
    'PetID':  petids_test,
    'y_true': y_test,
    'y_proba': y_test_proba,
})

print("val_predictions:")
print(val_predictions.head())
print(f"shape: {val_predictions.shape}")
print("\ntest_predictions:")
print(test_predictions.head())
print(f"shape: {test_predictions.shape}")

# 3. Tabela comparativa dos três modelos
comparison = pd.DataFrame({
    'model':         ['v1', 'v2', 'v3'],
    'architecture':  ['(128, 64)', '(64, 32)', '(32,)'],
    'alpha':         [0.0001, 0.01, 0.05],
    'n_iter':        [mlp.n_iter_, mlp_v2.n_iter_, mlp_v3.n_iter_],
    'roc_auc_train': [0.9588, 0.6849, round(train_roc_v3, 4)],
    'roc_auc_val':   [0.5312, 0.5464, round(val_roc_v3, 4)],
    'gap':           [0.4275, 0.1385, round(train_roc_v3 - val_roc_v3, 4)],
    'is_final':      [False, False, True],
})

print("\nTabela comparativa dos três modelos:")
print(comparison.to_string(index=False))

val_predictions:
       PetID  y_true   y_proba  y_pred
0  8e76c8e39       1  0.498617       0
1  c02be41e6       1  0.477808       0
2  85fc3c314       1  0.539724       1
3  eabb13cea       1  0.457098       0
4  0d25f084f       1  0.523678       1
shape: (2396, 4)

test_predictions:
       PetID  y_true   y_proba
0  86e1089a3       1  0.503023
1  3422e4906       0  0.452633
2  7a0942d61       0  0.472676
3  b10e7605a       0  0.576520
4  6436c1a59       1  0.420869
shape: (2997, 3)

Tabela comparativa dos três modelos:
model architecture  alpha  n_iter  roc_auc_train  roc_auc_val    gap  is_final
   v1    (128, 64) 0.0001      28         0.9588       0.5312 0.4275     False
   v2     (64, 32) 0.0100      16         0.6849       0.5464 0.1385     False
   v3        (32,) 0.0500      12         0.6008       0.5469 0.0539      True


In [15]:
# ============================================================
# Célula 11 — Persistir modelos + previsões + métricas + metadata
# ============================================================
import joblib

# 1. Guardar os três modelos (v3 é o oficial; v1 e v2 ficam como documentação)
joblib.dump(mlp,    RESULTS_DIR / "mlp_v1.joblib")
joblib.dump(mlp_v2, RESULTS_DIR / "mlp_v2.joblib")
joblib.dump(mlp_v3, RESULTS_DIR / "mlp_v3.joblib")
print(f"✓ Modelos guardados em: {RESULTS_DIR}")

# 2. Previsões (formato idêntico ao NB03 para facilitar NB06)
val_predictions.to_parquet(RESULTS_DIR / "val_predictions.parquet", index=False)
test_predictions.to_parquet(RESULTS_DIR / "test_predictions.parquet", index=False)
print(f"✓ Previsões guardadas (val + test)")

# 3. Tabela comparativa em CSV (legível num editor de texto ou Excel)
comparison.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)
print(f"✓ Tabela comparativa guardada")

# 4. Métricas do modelo oficial (v3) em JSON — fácil de ler programaticamente
metrics_final = {
    'notebook': 'NB04',
    'baseline_type': 'text',
    'final_model': 'v3',
    'architecture': {
        'input_dim': int(X_train.shape[1]),
        'hidden_layers': [32],
        'activation': 'relu',
        'alpha_l2': 0.05,
    },
    'training': {
        'n_iter': int(mlp_v3.n_iter_),
        'roc_auc_train': round(float(train_roc_v3), 4),
    },
    'validation_metrics': {k: round(float(v), 4) for k, v in metrics_val_v3.items()},
    'comparison_vs_nb03': {
        'nb03_tabular_roc_auc': 0.6758,
        'nb04_text_roc_auc':    round(float(val_roc_v3), 4),
        'difference':           round(float(val_roc_v3 - 0.6758), 4),
        'conclusion':           'Tabular supera texto isolado em ~0.13 ROC-AUC; confirma hipótese multimodal.',
    },
    'key_findings': [
        'Testadas 3 configuracoes de MLP com regularizacao progressivamente mais forte.',
        'v1 (alpha=1e-4) sofreu overfitting severo: train 0.96 vs val 0.53, gap +0.43.',
        'v2 (alpha=1e-2) e v3 (alpha=5e-2) convergiram ambas para ROC-AUC val ~0.55.',
        'Teto do sinal generalizavel do texto isolado estimado em ~0.55 ROC-AUC.',
        'Hipotese: sinal no texto e fortemente dependente do estilo do RescuerID, que nao generaliza entre splits.',
        'v3 escolhido como final por ter o menor gap train-val (+0.05) -> mais robusto para a fusao no NB06.',
    ],
    'notes_for_nb06': [
        'Usar test_predictions.parquet[y_proba] como feature de entrada do ramo texto.',
        'Nao usar y_pred de val_predictions.parquet: threshold deve ser decidido apos fusao.',
        'Modelo mlp_v3.joblib pode ser recarregado com joblib.load() se necessario.',
    ],
}

with open(RESULTS_DIR / "metrics.json", 'w', encoding='utf-8') as f:
    json.dump(metrics_final, f, indent=2, ensure_ascii=False)
print(f"✓ metrics.json guardado")

# 5. Listar tudo o que foi criado
print(f"\n📁 Conteúdo de {RESULTS_DIR}:")
for item in sorted(RESULTS_DIR.iterdir()):
    size_kb = item.stat().st_size / 1024
    print(f"  {item.name:35s}  {size_kb:>8.1f} KB")

✓ Modelos guardados em: /Users/rennandamiani/Documents/ISAG/ProjetoFinal_PetFinder/results/nb04_baseline_texto
✓ Previsões guardadas (val + test)
✓ Tabela comparativa guardada
✓ metrics.json guardado

📁 Conteúdo de /Users/rennandamiani/Documents/ISAG/ProjetoFinal_PetFinder/results/nb04_baseline_texto:
  metrics.json                              1.5 KB
  mlp_v1.joblib                           683.5 KB
  mlp_v2.joblib                           321.7 KB
  mlp_v3.joblib                           152.5 KB
  model_comparison.csv                      0.2 KB
  test_predictions.parquet                 51.9 KB
  val_predictions.parquet                  42.7 KB
